In [ ]:
ENV["OMP_NUM_THREADS"] = "4"

In [ ]:
using Pkg
Pkg.activate(".")
Pkg.develop(path="..")

isCuda = try
    success(`nvidia-smi`)
catch
    false
end
if isCuda
    println("CUDA is available, using GPU acceleration.")
    using CUDA
end

using bslLD, Random
bslLD.greet()

if isCuda
    println("Setting backend to CUDA.")
    bslLD.use_cuda!()
else
    println("CUDA not available, using CPU.")
end


In [ ]:
mutable struct Diag
    Ey :: Vector
    Bz :: Vector
    t  :: Vector{Float64}
end
Diag() = Diag([], [], Float64[])

function record!(d, sol, simTime)
    simTime.step % 1 == 0 || return
    push!(d.Ey, copy(Array(sol.E[2].data)))
    push!(d.Bz, copy(Array(sol.Bhalf[3].data)))
    push!(d.t,  simTime.current_T)
end

In [ ]:
compute_J_perp(f, grid, simTime) = bslLD.compute_current(f, grid, simTime.phase)

function stepStrang!(f_i, sol, grid, simTime, solver)

    phase_start = simTime.phase
    Ω  = simTime.gyro_frequency
    dt = simTime.dt

    n_i    = bslLD.compute_density(f_i, grid)
    J_perp = compute_J_perp(f_i, grid, simTime)
    moments = bslLD.Moments(n_i, J_perp, Pi_zero)

    # Both solvers: sol.Enew = E^{n+1}, sol.Bhalf = B_next, sol.E = E^n untouched
    bslLD.solve_fields!(sol, moments, grid, solver, dt)

    # Centered electric field E^{n+1/2}
    sol.Ecenter .= (sol.E + sol.Enew) * 0.5

    simTime.phase = phase_start;              simTime.fraction_dt = 0.5
    bslLD.advectV!(f_i, grid, simTime, sol.Ecenter)
    simTime.phase = phase_start + 0.5 * Ω * dt; simTime.fraction_dt = 1.0
    bslLD.advectX!(f_i, grid, simTime)
    simTime.phase = phase_start + Ω * dt;    simTime.fraction_dt = 0.5
    bslLD.advectV!(f_i, grid, simTime, sol.Ecenter)

    sol.E .= sol.Enew

    simTime.phase = phase_start
    simTime.fraction_dt = 1.0
end


In [ ]:
# k = (kx, 0, 0),  B0 = z-hat  (grid.Bdir = 3)
# Only Bz and (Ex, Ey) evolve; Ez = 0 and Pi_diff_z = 0 exactly for this geometry.

beta_i  = 0.1
mu      = 0.05
epsilon = 1e-7
T       = .1

Lx   = 50pi
Nx   = 128
Nv   = 32
vmax = 6 * sqrt(T)

dt = 0.01
Tmax = 20

grid    = bslLD.Grid([0.0, -vmax, -vmax], [Lx, vmax, vmax], [Nx, Nv, Nv], 1, 1.0, 3)
Pi_zero = bslLD.zero_vectorfield3(grid)

function make_ics(grid)
    Ey_rand = epsilon .* randn(Nx)
    E0 = bslLD.VectorField([
        bslLD.ScalarField(zeros(Nx)),
        bslLD.ScalarField(Ey_rand),
        bslLD.ScalarField(zeros(Nx)),
    ])
    B0 = bslLD.VectorField([bslLD.ScalarField(zeros(Nx)) for _ in 1:3])
    sol = bslLD.FieldSolution(E0, B0, bslLD.background_field(grid))
    initFuncv(v) = exp(-(v+epsilon*rand())^2 / (2T)) / sqrt(2pi)
    f_i = bslLD.Distribution(grid, 0.00001,
                              initFuncx = x -> 1.0,
                              initFuncv = initFuncv)
    return f_i, sol
end


In [ ]:
Random.seed!(42)
f_i, sol = make_ics(grid)
solver_si  = bslLD.SemiImplicitEMSolver(beta_i, mu)
simTime_si = bslLD.SimulationTime(dt, Tmax)
bslLD.initialize_Bhalf!(sol, grid, simTime_si.dt)
bslLD.ProgressMeter.ijulia_behavior(:clear)
diag_si = Diag()
while bslLD.continue_advection(simTime_si, true)
    stepStrang!(f_i, sol, grid, simTime_si, solver_si)
    bslLD.advance!(simTime_si)
    record!(diag_si, sol, simTime_si)
end


In [ ]:
Random.seed!(42)
f_i, sol = make_ics(grid)
solver_fi  = bslLD.FullyImplicitEMSolver(beta_i, mu)
simTime_fi = bslLD.SimulationTime(dt, Tmax)
bslLD.ProgressMeter.ijulia_behavior(:clear)
diag_fi = Diag()
while bslLD.continue_advection(simTime_fi, true)
    stepStrang!(f_i, sol, grid, simTime_fi, solver_fi)
    bslLD.advance!(simTime_fi)
    record!(diag_fi, sol, simTime_fi)
end


In [ ]:
using FFTW, DSP, CairoMakie

dt_diag = 0.02
omega   = fftfreq(length(diag_si.t), 1 / dt_diag) .* 2pi
k       = fftfreq(Nx, 1 / grid.delta[1]) .* 2pi

nw    = length(diag_si.t) ÷ 5
nk    = Nx ÷ 2
w_win = kaiser(length(diag_si.t), 3)

function make_spec(data_vec)
    data     = transpose(hcat(data_vec...))
    windowed = data .* w_win
    log.(abs.(fft(windowed))[1:nw, 1:nk] .+ 1e-30)
end

fig = Figure(size = (1200, 500))
for (col, (d, label)) in enumerate([(diag_si, "SemiImplicit"), (diag_fi, "FullyImplicit")])
    ax = Axis(fig[1, 2col-1], xlabel = "k", ylabel = "ω", title = "Ey spectrum — $label")
    hm = heatmap!(ax, k[1:nk], omega[1:nw], make_spec(d.Ey)')
    Colorbar(fig[1, 2col], hm)
end
fig


In [ ]:
using Statistics

fig = Figure()
ax  = Axis(fig[1, 1], xlabel = "t", ylabel = "⟨Ey²⟩", title = "Ey energy comparison")
lines!(ax, diag_si.t, map(x -> mean(x .^ 2), diag_si.Ey), label = "SemiImplicit")
lines!(ax, diag_fi.t, map(x -> mean(x .^ 2), diag_fi.Ey), label = "FullyImplicit")
axislegend(ax)
fig
